# 图像编码 · 教师版 Notebook

配合 `course-design.qmd` 与 `image-encoding-v1.pptx` 使用。教师版包含素材生成、fallback 导出、接缝验证和 BMP 复核。

## 第一格 · 路径检查

必须从 `1-2-encoding/1-2-3-image-encoding` 目录启动 Jupyter。

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
import struct

lesson_dir = Path.cwd()
assets_dir = lesson_dir / "assets"

# 推荐从本课目录启动；若 Jupyter 自动把 cwd 设为 notebooks/，则自动回到上一级。
if not assets_dir.is_dir() and lesson_dir.name == "notebooks" and (lesson_dir.parent / "assets").is_dir():
    lesson_dir = lesson_dir.parent
    assets_dir = lesson_dir / "assets"
    import os
    os.chdir(lesson_dir)

assert assets_dir.is_dir(), (
    "当前 Jupyter working directory 不是本课目录：" + str(Path.cwd()) + "\n"
    "请从 1-2-encoding/1-2-3-image-encoding 目录启动 notebook，或从 notebooks/ 打开后允许自动切换到上一级。"
)
print("lesson directory:", lesson_dir)
print("assets directory:", assets_dir)

## Q1 / Q3 · 生成固定测试图

预测：哪些细节会在低采样时先消失？红绿接缝在颜色数量减少时会怎样？

In [ ]:
# Q1/Q3 共用固定测试图：无随机数，每次结果一致
path = Path("assets/test-image.png")
path.parent.mkdir(parents=True, exist_ok=True)
w = h = 256
img = Image.new("RGB", (w, h))
px = img.load()
for y in range(h):
    for x in range(w):
        v = round(255 * x / (w - 1))
        px[x, y] = (v, v, v)

draw = ImageDraw.Draw(img)
draw.rectangle((12, 12, 72, 72), fill=(255, 0, 0))
draw.rectangle((73, 12, 133, 72), fill=(0, 255, 0))
draw.rectangle((184, 12, 244, 72), fill=(0, 0, 255))
draw.rectangle((0, 104, 128, 255), fill=(224, 224, 224))
draw.rectangle((136, 104, 255, 255), fill=(224, 224, 224))
draw.ellipse((0, 112, 128, 240), outline=(0, 0, 0), width=28)
draw.line((144, 240, 240, 144), fill=(0, 0, 0), width=28)
for x in [152, 160, 168, 176, 184, 192]:
    draw.line((x, 224, x, 244), fill=(255, 255, 255), width=2)
img.save(path)
assert img.size == (256, 256)
assert img.mode == "RGB"
print("created:", path)
display(img)

## Q1 · Demo A：Sampling

备课时导出固定五张图；课堂默认使用 PPTX 两页证据页。若学生提出新尺寸，用 `show_sampling(n)` 即时验证。失败时直接使用 PPTX fallback。

In [ ]:
# Q1 · Demo A：备课导出固定 sampling 图片；课堂即时验证请用 show_sampling(n)
source_path = Path("assets/test-image.png")
assert source_path.is_file()
src = Image.open(source_path).convert("RGB")

sampling_outputs = {}

def make_sampling(n):
    sampled = src.resize((n, n), resample=Image.Resampling.NEAREST)
    enlarged = sampled.resize((512, 512), resample=Image.Resampling.NEAREST)
    assert sampled.mode == "RGB"
    assert enlarged.mode == "RGB"
    assert enlarged.size == (512, 512)
    return enlarged

for n in [256, 64, 32, 16, 8]:
    sampling_outputs[n] = make_sampling(n)
    print(f"{n} × {n}")
    display(sampling_outputs[n])

def show_sampling(n: int):
    """课堂即时验证：输入学生提出的新尺寸，例如 show_sampling(24)。"""
    assert isinstance(n, int) and n > 0
    img = make_sampling(n)
    print(f"{n} × {n}")
    display(img)
    return img

## Q2 · 分辨率不是质量的全部

检查两张 Q2 图片的真实尺寸。

In [ ]:
# Q2 · 文件尺寸复核：高像素数但模糊 vs 较低像素数但清晰
for p in [Path("assets/fallback/q2-high-pixels-blur.png"), Path("assets/fallback/q2-low-pixels-sharp.png")]:
    assert p.is_file()
    with Image.open(p) as im:
        print(p.name, im.size)
assert Image.open("assets/fallback/q2-high-pixels-blur.png").size == (1920, 1080)
assert Image.open("assets/fallback/q2-low-pixels-sharp.png").size == (800, 600)

## Q3 · Demo B：Quantization

学生先看图解释；不要提前打印接缝 RGB。

In [ ]:
# Q3 · Demo B：主课只看 original / 16-color / 2-color；先看图，后揭示接缝数值
source_path = Path("assets/test-image.png")
assert source_path.is_file()
src = Image.open(source_path).convert("RGB")
targets = [256, 16, 4, 2]
main_targets = [16, 2]
outputs = {}

for n in targets:
    q = src.quantize(colors=n, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE).convert("RGB")
    colors = q.getcolors(maxcolors=n + 1)
    assert colors is not None
    color_count = len(colors)
    assert q.size == src.size
    assert q.mode == "RGB"
    assert color_count <= n
    outputs[n] = q

print("original")
display(src)
for n in main_targets:
    print(f"max {n} colors")
    display(outputs[n])

## Q3 · 揭示接缝证据

学生观察并解释后，再运行这一格揭示接缝两侧的 RGB tuple。

In [ ]:
# Q3 · 接缝证据：学生观察解释之后再运行/揭示
seam_left = (72, 40)
seam_right = (73, 40)
for n in [16, 2]:
    left_rgb = outputs[n].getpixel(seam_left)
    right_rgb = outputs[n].getpixel(seam_right)
    print(f"{n:2d} colors | left={left_rgb} | right={right_rgb} | same={left_rgb == right_rgb}")

## 备课导出 fallback

这一步保证 PPTX、live demo、故障备用图来自同一批证据。

In [ ]:
# 备课导出 fallback 图片：PPTX、live demo、故障备用图使用同一批证据
fallback_dir = Path("assets/fallback")
fallback_dir.mkdir(parents=True, exist_ok=True)
for n in [256, 64, 32, 16, 8]:
    img = sampling_outputs[n]
    assert img.mode == "RGB" and img.size == (512, 512)
    img.save(fallback_dir / f"sampling-{n}.png")

quant_images = {"original": src, "16": outputs[16], "2": outputs[2]}
for label, img in quant_images.items():
    assert img.mode == "RGB" and img.size == (256, 256)
    img.save(fallback_dir / f"quant-{label}.png")
print("fallback images exported to:", fallback_dir)

In [ ]:
# 备课复核：重新打开导出的文件检查尺寸、mode 与红绿接缝
for n in [256, 64, 32, 16, 8]:
    p = Path("assets/fallback") / f"sampling-{n}.png"
    with Image.open(p) as im:
        assert im.mode == "RGB"
        assert im.size == (512, 512)

for label in ["original", "16", "2"]:
    p = Path("assets/fallback") / f"quant-{label}.png"
    with Image.open(p) as im:
        assert im.mode == "RGB"
        assert im.size == (256, 256)

seam_left = (72, 40)
seam_right = (73, 40)
with Image.open("assets/fallback/quant-16.png").convert("RGB") as q16:
    assert q16.getpixel(seam_left) != q16.getpixel(seam_right)
with Image.open("assets/fallback/quant-2.png").convert("RGB") as q2:
    assert q2.getpixel(seam_left) == q2.getpixel(seam_right)
print("fallback verification passed")

## Q7 / Q11 · BMP 文件证据

In [ ]:
# Q7/Q11 · BMP 证据复核
path = Path("assets/q7-3x2-24bit.bmp")
assert path.is_file()
data = path.read_bytes()
pixel_offset = struct.unpack_from("<I", data, 10)[0]
width = struct.unpack_from("<i", data, 18)[0]
height = struct.unpack_from("<i", data, 22)[0]
bpp = struct.unpack_from("<H", data, 28)[0]
rows = abs(height)
color_value_bytes = width * rows * bpp // 8
raw_row_bytes = width * bpp // 8
stored_row_bytes = ((width * bpp + 31) // 32) * 4
pixel_array_bytes = stored_row_bytes * rows
print("pixel-data offset :", pixel_offset, "bytes")
print("color values      :", color_value_bytes, "bytes")
print("raw row values    :", raw_row_bytes, "bytes/row")
print("stored row size   :", stored_row_bytes, "bytes/row")
print("pixel array       :", pixel_array_bytes, "bytes")
print("whole file        :", len(data), "bytes")
assert pixel_offset == 54
assert color_value_bytes == 18
assert stored_row_bytes == 12
assert pixel_array_bytes == 24
assert len(data) == 78